In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

In [ ]:
resize = 1
plt.rcParams.update({
    "figure.figsize": (6.4*resize, 4.0*resize), # (6.4, 4.8)[4:3] -> (6.4, 4.0)[8:5]
    "font.sans-serif": ["Helvetica", "Nimbus Sans", "Arial", "DejaVu Sans"],
})

In [ ]:
def get_npy(file, ranges):
    npy = np.load(file, allow_pickle=True).item()
    for key, value in npy.items():
        npy[key] = value[:ranges]
    return npy

In [ ]:
ranges = 105 * 4

npy_uniform = get_npy("./npy/imagenet_resnet18_e105_t105_w5s4_amp_schedule-uniform.npy", ranges)
npy_no_cycle = get_npy("./npy/imagenet_resnet18_e105_t105_w5s4_amp_schedule-no-cycle.npy", ranges)
npy_cyclic = get_npy("./npy/imagenet_resnet18_e105_t105_w5s4_amp_schedule-cyclic.npy", ranges)

In [ ]:
print(npy_cyclic.keys())

In [ ]:
SAVE = True
metadata = "pdf"

x_range = np.arange(ranges) / 4

fig, ax1 = plt.subplots()
plt.xlabel("Epoch")
plt.gca().xaxis.set_major_locator(MultipleLocator(10))

ax1.set_ylabel("Validation Loss (Solid)")
ax1.plot(x_range, npy_uniform["val_loss"], label="uniform")
ax1.plot(x_range, npy_no_cycle["val_loss"], label="w/o cycle")
ax1.plot(x_range, npy_cyclic["val_loss"], label="w cycle")

ax2 = ax1.twinx()
ax2.set_ylabel("Validation Accuracy (Dashed)")
ax2.plot(x_range, npy_uniform["val_acc"], "--", label="uniform")
ax2.plot(x_range, npy_no_cycle["val_acc"], "--", label="w/o cycle")
ax2.plot(x_range, npy_cyclic["val_acc"], "--", label="w cycle")

ax1.legend(loc="right")
plt.savefig(f"impact_cycle.{metadata}", bbox_inches="tight") if SAVE else plt.show()

|  | uniform | w/o cycle | w cycle |
|:-:|-:|-:|-:|
| loss | 1.545 | 1.492 | 1.404 |
| accuracy | 62.6% | 64.0% | 65.8% |